# Практика · 07. Згортковий шар

> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) · ДЗ: [homework.html](homework.html)

Тут ми розберемо згортковий шар на числа й переконаємось, що всередині `nn.Conv2d`
немає нічого, чого не можна написати самому.

Що зробимо:

1. згенеруємо датасет фігур 28 × 28 формулами — нічого не завантажуючи;
2. напишемо **власну згортку на numpy** й доведемо через `np.allclose`, що вона
   збігається з `nn.Conv2d` при однакових вагах;
3. порахуємо параметри шару **руками** й звіримо із `sum(p.numel())`;
4. порахуємо форму виходу **за формулою** й звіримо з фактичним `.shape`;
5. доведемо одиничним імпульсом, що `nn.Conv2d` рахує **кореляцію**, а не згортку;
6. навчимо маленьку мережу й подивимось на ваги її першого шару.

> **Мережа не потрібна:** датасет ми малюємо самі, формулами.
>
> ⏱ Зошит навчає одну маленьку мережу. Заміряно: близько хвилини
> на чотирьох ядрах без відеокарти.

In [ ]:
import sys, time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# Зерна фіксуємо на початку: без цього числа в лекції й у тебе розійдуться.
torch.manual_seed(0)
torch.set_num_threads(4)

print("Python     :", sys.version.split()[0])
print("numpy      :", np.__version__)
print("torch      :", torch.__version__)
print("пристрій   : CPU (GPU тут не потрібен)")

## 1. Датасет: фігури 28 × 28, намальовані формулами

Шість класів — коло, квадрат, ромб, кільце, хрест, трикутник. Центр кожної фігури
трохи зсунуто, розмір трохи змінюється, зверху накладено шум. Нічого не
завантажується: усе рахується з координатної сітки.

In [ ]:
def make_shapes(n_per_class, classes, shift=3, noise=0.08, seed=42):
    """Малює фігури 28×28 формулами. Повертає (images, labels) у форматі (N, 1, 28, 28)."""
    rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[0:28, 0:28]          # координати кожного пікселя
    images, labels = [], []
    for label, name in enumerate(classes):
        for _ in range(n_per_class):
            # зсув центра й розмір беремо випадково, щоб фігури не були однаковими
            cy = 13.5 + rng.integers(-shift, shift + 1)
            cx = 13.5 + rng.integers(-shift, shift + 1)
            radius = rng.uniform(6.5, 9.0)
            dy, dx = yy - cy, xx - cx
            if name == "коло":
                mask = dy * dy + dx * dx <= radius * radius
            elif name == "квадрат":
                mask = (np.abs(dy) <= radius * 0.85) & (np.abs(dx) <= radius * 0.85)
            elif name == "ромб":
                mask = np.abs(dy) + np.abs(dx) <= radius * 1.15
            elif name == "кільце":
                dist = np.sqrt(dy * dy + dx * dx)
                mask = (dist <= radius) & (dist >= radius - 2.5)
            elif name == "хрест":
                mask = ((np.abs(dy) <= radius) & (np.abs(dx) <= 2.0)) | \
                       ((np.abs(dx) <= radius) & (np.abs(dy) <= 2.0))
            else:  # трикутник
                mask = (dy <= radius * 0.8) & (dy >= -radius * 0.8) & \
                       (np.abs(dx) <= (dy + radius * 0.8) * 0.62)
            picture = mask.astype(np.float32) + rng.normal(0, noise, (28, 28)).astype(np.float32)
            images.append(np.clip(picture, 0, 1))
            labels.append(label)
    images = np.stack(images)[:, None]     # додаємо вісь каналу: (N, 1, 28, 28)
    labels = np.array(labels, dtype=np.int64)
    order = rng.permutation(len(labels))   # перемішуємо, щоб класи не йшли блоками
    return images[order], labels[order]


SHAPE_NAMES = ["коло", "квадрат", "ромб", "кільце", "хрест", "трикутник"]

start = time.time()
train_images, train_labels = make_shapes(200, SHAPE_NAMES, seed=42)
test_images, test_labels = make_shapes(50, SHAPE_NAMES, seed=7)
print(f"згенеровано за {time.time() - start:.2f} с")
print("навчальна вибірка:", train_images.shape, train_images.dtype)
print("перевірочна      :", test_images.shape)

Подивимось на шість фігур — по одній кожного класу.

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(11, 2))
for class_index, ax in enumerate(axes):
    # беремо перший приклад цього класу з навчальної вибірки
    first = np.where(train_labels == class_index)[0][0]
    ax.imshow(train_images[first, 0], cmap="gray")
    ax.set_title(SHAPE_NAMES[class_index], fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()
print("шість класів, по", (train_labels == 0).sum(), "прикладів кожного")

## 2. Форма даних: `(N, C, H, W)`

PyTorch чекає чотиривимірний тензор: пачка, канали, висота, ширина. Подивимось,
що шар робить із формою.

In [ ]:
demo_layer = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
demo_batch = torch.from_numpy(train_images[:8])          # 8 картинок із датасету

print("вхід :", tuple(demo_batch.shape), " ← (N, C, H, W)")
print("вихід:", tuple(demo_layer(demo_batch).shape), " ← каналів стало 16, розмір той самий")
print()
print("вага шару має форму:", tuple(demo_layer.weight.shape),
      " ← (out_channels, in_channels, k, k)")
print("зсув шару має форму:", tuple(demo_layer.bias.shape), " ← по одному числу на фільтр")

## 3. Власна згортка на numpy проти `nn.Conv2d`

Найважливіша перевірка теми. Пишемо згортку чотирма вкладеними циклами — так, як
її пояснює лекція, — кладемо ті самі ваги в `nn.Conv2d` і порівнюємо результати
через `np.allclose`.

Зверни увагу на порядок індексів у вікні: ми беремо `image[y + i, x + j]`, тобто
індекс ядра **додається** до індексу зображення. Це кореляція, і саме її рахує
PyTorch — доведемо це окремо в розділі 6.

In [ ]:
def my_conv2d(image, kernel, bias, stride=1, padding=0):
    """Згортковий шар на чистому numpy. image: (N, C, H, W), kernel: (O, C, k, k)."""
    n_images, n_channels, height, width = image.shape
    n_filters, kernel_channels, kh, kw = kernel.shape
    assert n_channels == kernel_channels, "ядро мусить мати стільки ж каналів, скільки вхід"

    if padding:
        # рамка з нулів — саме те, що nn.Conv2d робить за замовчуванням
        image = np.pad(image, ((0, 0), (0, 0), (padding, padding), (padding, padding)))

    out_h = (height + 2 * padding - kh) // stride + 1
    out_w = (width + 2 * padding - kw) // stride + 1
    result = np.zeros((n_images, n_filters, out_h, out_w), dtype=np.float32)

    for n in range(n_images):
        for f in range(n_filters):
            for row in range(out_h):
                for col in range(out_w):
                    top, left = row * stride, col * stride
                    # вікно захоплює ВСІ канали одразу — саме тому ядро має глибину
                    window = image[n, :, top:top + kh, left:left + kw]
                    result[n, f, row, col] = np.sum(window * kernel[f]) + bias[f]
    return result


print("функція готова: чотири цикли, жодної магії")

In [ ]:
rng = np.random.default_rng(42)
# маленький випадковий вхід і випадкові ваги — щоб перевірка не залежала від датасету
sample_input = rng.random((2, 3, 9, 9)).astype(np.float32)
sample_weight = (rng.standard_normal((4, 3, 3, 3)) * 0.3).astype(np.float32)
sample_bias = (rng.standard_normal(4) * 0.1).astype(np.float32)

print(f"{'stride':>7}{'padding':>9}{'наша форма':>16}{'torch форма':>16}{'max різниця':>14}")
for stride, padding in [(1, 0), (1, 1), (2, 1), (3, 2)]:
    ours = my_conv2d(sample_input, sample_weight, sample_bias, stride, padding)

    torch_layer = nn.Conv2d(3, 4, 3, stride=stride, padding=padding)
    with torch.no_grad():                       # ваги підміняємо, а не вчимо
        torch_layer.weight.copy_(torch.from_numpy(sample_weight))
        torch_layer.bias.copy_(torch.from_numpy(sample_bias))
    theirs = torch_layer(torch.from_numpy(sample_input)).detach().numpy()

    assert np.allclose(ours, theirs, atol=1e-5), "розрахунок розійшовся!"
    diff = np.abs(ours - theirs).max()
    print(f"{stride:>7}{padding:>9}{str(ours.shape):>16}{str(theirs.shape):>16}{diff:>14.2e}")

print()
print("✅ власна згортка збігається з nn.Conv2d на всіх чотирьох наборах")

## 4. Скільки в шарі параметрів

Формула з лекції: `out × in × k × k + out`. Перший доданок — коефіцієнти ядер,
другий — зсуви, по одному на фільтр. Порахуємо руками й звіримо із `sum(p.numel())`.

In [ ]:
print(f"{'шар':<26}{'руками':>10}{'numel':>10}   збіг")
for in_ch, out_ch, k in [(1, 8, 3), (3, 16, 3), (3, 16, 5), (3, 32, 7), (64, 64, 3), (256, 256, 1)]:
    layer = nn.Conv2d(in_ch, out_ch, k)
    by_hand = out_ch * in_ch * k * k + out_ch
    by_torch = sum(p.numel() for p in layer.parameters())
    assert by_hand == by_torch, "формула розійшлася з бібліотекою!"
    name = f"Conv2d({in_ch}, {out_ch}, {k})"
    print(f"{name:<26}{by_hand:>10}{by_torch:>10}   ✅")

Тепер найважливіше в цій формулі — те, чого в ній **немає**. Розмір зображення
на кількість параметрів не впливає взагалі: ядро одне на всю картинку.

In [ ]:
size_layer = nn.Conv2d(3, 16, 3, padding=1)
n_params = sum(p.numel() for p in size_layer.parameters())

for side in (28, 128, 512):
    output_shape = tuple(size_layer(torch.zeros(1, 3, side, side)).shape)
    print(f"вхід {side:>3} × {side:<3} → вихід {str(output_shape):<20} параметрів: {n_params}")

print()
# для порівняння: повнозвʼязний шар на тому самому перетворенні при 28 × 28
dense_in, dense_out = 3 * 28 * 28, 16 * 28 * 28
dense_params = dense_in * dense_out + dense_out
print(f"той самий перехід повнозвʼязним шаром при 28 × 28: {dense_params:,} параметрів"
      .replace(",", " "))
print(f"це більше в {dense_params // n_params:,} рази — і залежить від розміру картинки"
      .replace(",", " "))

## 5. Форма виходу: формула проти факту

`H_out = (H + 2p − k) // s + 1`, де `//` — ділення з округленням униз. Порахуємо
для семи наборів і звіримо з тим, що насправді видасть шар.

Окремо порахуємо, скільки справжніх пікселів **не потрапило в жодне вікно**: коли
`(H + 2p − k)` не ділиться на `s` націло, край мовчки відрізається.

In [ ]:
def uncovered_pixels(H, k, s, p):
    """Скільки крайніх пікселів не побачив жоден фільтр."""
    out = (H + 2 * p - k) // s + 1
    covered = np.zeros(H, dtype=bool)
    for step in range(out):
        left = step * s                     # ліва межа вікна в координатах із рамкою
        for position in range(left, left + k):
            real = position - p             # переводимо назад у координати зображення
            if 0 <= real < H:
                covered[real] = True
    return int((~covered).sum())


print(f"{'H':>4}{'k':>4}{'s':>4}{'p':>4}{'формула':>10}{'факт':>7}{'обрізано':>11}")
for H, k, s, p in [(28, 3, 1, 0), (28, 3, 1, 1), (28, 5, 1, 2), (28, 3, 2, 1),
                   (28, 3, 2, 0), (28, 7, 2, 3), (28, 4, 3, 0), (10, 3, 3, 0)]:
    by_formula = (H + 2 * p - k) // s + 1
    layer = nn.Conv2d(1, 1, k, stride=s, padding=p)
    actual = layer(torch.zeros(1, 1, H, H)).shape[-1]
    assert by_formula == actual, "формула розійшлася з фактичною формою!"
    print(f"{H:>4}{k:>4}{s:>4}{p:>4}{by_formula:>10}{actual:>7}{uncovered_pixels(H, k, s, p):>11}")

print()
print("✅ формула збігається з .shape на всіх восьми наборах")

## 6. `nn.Conv2d` рахує кореляцію, а не згортку

Найкоротший доказ — подати на вхід чорний квадрат з єдиною білою точкою. Справжня
згортка поверне ядро як є, кореляція — ядро, **повернуте на 180°**.

In [ ]:
kernel_123 = np.arange(1, 10, dtype=np.float32).reshape(3, 3)
impulse = np.zeros((1, 1, 7, 7), dtype=np.float32)
impulse[0, 0, 3, 3] = 1.0                    # єдина біла точка рівно в центрі

impulse_layer = nn.Conv2d(1, 1, 3, padding=1, bias=False)
with torch.no_grad():
    impulse_layer.weight.copy_(torch.from_numpy(kernel_123.reshape(1, 1, 3, 3)))
response = impulse_layer(torch.from_numpy(impulse)).detach().numpy()[0, 0, 2:5, 2:5]

print("ядро, яке ми поклали у ваги:")
print(kernel_123)
print()
print("відгук на одиничний імпульс:")
print(response)
print()
print("відгук == ядро як є          :", np.allclose(response, kernel_123))
print("відгук == ядро, перевернуте  :", np.allclose(response, np.flip(kernel_123)))
print()
print("⚠️ отже, nn.Conv2d рахує КОРЕЛЯЦІЮ: щоб дістати справжню згортку,")
print("   ядро треба перевернути самому через np.flip перед тим, як класти у ваги")

Для **навченого** ядра різниці немає: мережа просто вивчить дзеркальний варіант.
Різниця важлива рівно тоді, коли ти кладеш у ваги готові коефіцієнти з підручника.

## 7. Рецептивне поле: два шари 3 × 3 проти одного 5 × 5

Обидва варіанти бачать ділянку 5 × 5 і дають той самий розмір виходу. Порівняймо
ціну.

In [ ]:
two_small = nn.Sequential(nn.Conv2d(16, 16, 3), nn.Conv2d(16, 16, 3))
one_wide = nn.Conv2d(16, 16, 5)

probe = torch.zeros(1, 16, 32, 32)
params_two = sum(p.numel() for p in two_small.parameters())
params_one = sum(p.numel() for p in one_wide.parameters())

print("два шари 3 × 3 :", params_two, "параметрів, вихід", tuple(two_small(probe).shape))
print("один шар 5 × 5 :", params_one, "параметрів, вихід", tuple(one_wide(probe).shape))
print()
print(f"стос дешевший на {100 * (1 - params_two / params_one):.0f} %"
      f" — і має ReLU між шарами, якої в одному ядрі немає")

## 8. Навчаємо мережу й дивимось на ваги першого шару

Головна ілюстрація теми. Мережа маленька: перший шар — вісім фільтрів 5 × 5
(208 параметрів), далі два шари з кроком 2 і лінійна голова. Ми **ніде** не кажемо
мережі, що таке межа — тільки показуємо картинки й назви фігур.

⚠️ Ця клітинка рахується найдовше в зошиті: близько 25 секунд на вільній машині
(і суттєво довше, якщо процесор чимось зайнятий).

In [ ]:
class SmallCNN(nn.Module):
    """Три згорткові шари й лінійна голова. Нічого зайвого."""

    def __init__(self, n_filters=8, n_classes=6):
        super().__init__()
        self.conv1 = nn.Conv2d(1, n_filters, 5, padding=2)          # 28 × 28 → 28 × 28
        self.conv2 = nn.Conv2d(n_filters, 8, 3, stride=2, padding=1)  # 28 → 14
        self.conv3 = nn.Conv2d(8, 8, 3, stride=2, padding=1)          # 14 → 7
        self.head = nn.Linear(8 * 7 * 7, n_classes)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = torch.relu(self.conv3(x))
        return self.head(x.flatten(1))


# зерно ставимо рівно тут: далі кожен виклик генератора має піти в тому самому порядку
torch.manual_seed(0)
model = SmallCNN()

print("усього параметрів :", sum(p.numel() for p in model.parameters()))
print("з них у conv1     :", sum(p.numel() for p in model.conv1.parameters()),
      "= 8 × 1 × 5 × 5 + 8")

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)
loss_function = nn.CrossEntropyLoss()
X = torch.from_numpy(train_images)
y = torch.from_numpy(train_labels)

start = time.time()
for epoch in range(12):
    order = torch.randperm(len(y))
    for batch_start in range(0, len(y), 32):
        batch = order[batch_start:batch_start + 32]
        optimizer.zero_grad()
        loss = loss_function(model(X[batch]), y[batch])
        loss.backward()
        optimizer.step()

elapsed = time.time() - start
with torch.no_grad():
    predicted = model(torch.from_numpy(test_images)).argmax(1).numpy()
accuracy = (predicted == test_labels).mean()

print(f"12 епох на {len(y)} прикладах — {elapsed:.1f} с")
print(f"точність на перевірочній вибірці: {accuracy:.3f}")

Дістаємо ваги першого шару. Для кожного з восьми фільтрів рахуємо три числа:
**суму ваг** (наскільки він реагує просто на яскравість), **ліво − право** (різницю
між двома лівими й двома правими стовпцями) і **верх − низ** (те саме по рядках).
Найбільше з трьох за модулем і каже, що фільтр шукає.

In [ ]:
weights = model.conv1.weight.detach().numpy()[:, 0]   # (8, 5, 5), вхід один канал

print(f"{'фільтр':>7}{'сума ваг':>11}{'ліво−право':>13}{'верх−низ':>11}   що це")
for index, filt in enumerate(weights):
    total = filt.sum()
    left_right = filt[:, :2].sum() - filt[:, 3:].sum()
    top_bottom = filt[:2, :].sum() - filt[3:, :].sum()

    big_edge = max(abs(left_right), abs(top_bottom))
    small_edge = min(abs(left_right), abs(top_bottom))
    if abs(total) >= big_edge:
        kind = "детектор плями" if total > 0 else "детектор плями навпаки"
    elif small_edge >= 0.8 * big_edge:
        # перепад є і по горизонталі, і по вертикалі — фільтр ловить кут
        kind = "похила межа (кут)"
    elif abs(top_bottom) > abs(left_right):
        kind = "горизонтальна межа"
    else:
        kind = "вертикальна межа"
    print(f"{index + 1:>7}{total:>11.2f}{left_right:>13.2f}{top_bottom:>11.2f}   {kind}")

print()
print("перший фільтр числами:")
print(np.round(weights[0], 3))

А тепер подивимось на ті самі ваги очима — і на те, як кожен фільтр
відгукується на одну фігуру.

In [ ]:
sample = train_images[np.where(train_labels == 1)[0][0]]      # квадрат
with torch.no_grad():
    responses = model.conv1(torch.from_numpy(sample[None])).numpy()[0]

fig, axes = plt.subplots(2, 9, figsize=(13, 3.4))
axes[0, 0].imshow(sample[0], cmap="gray"); axes[0, 0].set_title("вхід", fontsize=8)
axes[1, 0].axis("off")
limit = np.abs(weights).max()
for index in range(8):
    axes[0, index + 1].imshow(weights[index], cmap="RdBu_r", vmin=-limit, vmax=limit)
    axes[0, index + 1].set_title(f"ваги {index + 1}", fontsize=8)
    axes[1, index + 1].imshow(responses[index], cmap="RdBu_r")
    axes[1, index + 1].set_title(f"відгук {index + 1}", fontsize=8)
for ax in axes.ravel():
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()
print("угорі — ваги 5 × 5, унизу — карта ознак того самого фільтра на квадраті")

Червоне — додатні ваги, синє — відʼємні. Видно, що фільтри різні: у частини
верхня половина одного знака, а нижня іншого (детектор горизонтальних меж), у
частини те саме зліва направо, а один майже весь однакового знака — він просто
міряє яскравість.

Чесно про межі демонстрації: фільтри вийшли **шумні**. Датасет із простих фігур
мережа розвʼязує майже до 100 % за кілька епох, після чого градієнт майже зникає й
ваги перестають рухатись. Красиві смугасті фільтри з відомих ілюстрацій — це перший
шар великих мереж, навчених на мільйоні різноманітних фотографій.

## Завдання

### 🟢 Рівень 1
Постав `n_filters=16` замість 8, перенавчи мережу й порахуй параметри `conv1`
руками **перед** запуском. Звір із `sum(p.numel())`.

### 🟡 Рівень 2
Додай у `my_conv2d` підтримку `dilation` (проміжків між елементами ядра) і звір
із `nn.Conv2d(..., dilation=2)` через `np.allclose`.

### 🔴 Рівень 3
Напиши згортковий шар класом із прямим **і зворотним** проходом і звір градієнти
по вагах, зсуву та входу з тим, що дає `nn.Conv2d`. Деталі — у
[homework.md](homework.html).